In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import get_peft_model, LoraConfig, TaskType, PeftModel

In [ ]:
class CodeGenMultiModelWrapper:

    def __init__(self, model_name: str = "Qwen/Qwen3-Coder-30B-A3B-Instruct", device: str = "cuda", use_lora: bool = True, lora_kwargs: dict = None):
        self.device = "cuda" if torch.cuda.is_available() and device == "cuda" else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        base_model = AutoModelForCausalLM.from_pretrained(model_name).to(self.device)

        if use_lora:
            cfg = lora_kwargs or {"r": 16, "lora_alpha": 32, "target_modules": ["qkv_proj"], "lora_dropout": 0.05}
            peft_config = LoraConfig(
                task_type=TaskType.CAUSAL_LM,
                r=cfg.get("r", 16),
                lora_alpha=cfg.get("lora_alpha", 32),
                lora_dropout=cfg.get("lora_dropout", 0.05),
                target_modules=cfg.get("target_modules", ["qkv_proj"])
            )
            self.model = get_peft_model(base_model, peft_config)
            self.model.print_trainable_parameters()
        else:
            self.model = base_model

    def generate_code(self, prompt: str, max_new_tokens: int = 64) -> str:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        outputs = self.model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=self.tokenizer.eos_token_id)
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def extract_embeddings(self, code_text: str):
        inputs = self.tokenizer(code_text, return_tensors="pt", truncation=True, max_length=512).to(self.device)
        with torch.no_grad():
            underlying = self.model.base_model.model if hasattr(self.model, "base_model") else self.model
            outputs = underlying(**inputs, output_hidden_states=True)
            embeddings = outputs.hidden_states[-1].mean(dim=1).squeeze().cpu().numpy()
        return embeddings.astype("float32")

    def save_lora_weights(self, output_dir: str):
        if hasattr(self.model, "save_pretrained"):
            self.model.save_pretrained(output_dir)
            self.tokenizer.save_pretrained(output_dir)